<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day9_10_(260415)_React_Query_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# React Query 캐싱 실습

In [ ]:
%%writefile react-query-practice/server/server.js

const express = require('express');
const cors = require('cors');

const app = express();
const PORT = 5000;

app.use(cors());
app.use(express.json());

app.get('/api/users', async function (req, res) {
  try {
    await new Promise(function (resolve) {
      setTimeout(resolve, 1500);
    });

    const currentTime = new Date().toLocaleTimeString('ko-KR');

    const users = [
      {
        id: 1,
        name: '김민수',
        department: '개발팀',
        updatedAt: currentTime
      },
      {
        id: 2,
        name: '이서연',
        department: '기획팀',
        updatedAt: currentTime
      },
      {
        id: 3,
        name: '박준호',
        department: '운영팀',
        updatedAt: currentTime
      }
    ];

    res.json(users);
  } catch (error) {
    res.status(500).json({
      message: '사용자 목록 조회 실패'
    });
  }
});

app.listen(PORT, function () {
  console.log('Node server is running on port ' + PORT);
});

Overwriting react-query-practice/server/server.js


In [ ]:
%%writefile react-query-practice/client/src/api/usersApi.js

export async function fetchUsers() {
  const response = await fetch('/api/users');

  if (!response.ok) {
    throw new Error('사용자 목록 조회 실패');
  }

  return await response.json();
}

Writing react-query-practice/client/src/api/usersApi.js


In [ ]:
%%writefile react-query-practice/client/src/main.jsx

import React from 'react';
import ReactDOM from 'react-dom/client';
import { QueryClient, QueryClientProvider } from '@tanstack/react-query';
import App from './App';
import './styles/app.css';

const queryClient = new QueryClient();

ReactDOM.createRoot(document.getElementById('root')).render(
  <React.StrictMode>
    <QueryClientProvider client={queryClient}>
      <App />
    </QueryClientProvider>
  </React.StrictMode>
);

Overwriting react-query-practice/client/src/main.jsx


In [ ]:
%%writefile react-query-practice/client/src/App.jsx

import { useQuery } from '@tanstack/react-query';
import { fetchUsers } from './api/usersApi';
import UserList from './components/UserList';
import UserSummary from './components/UserSummary';
import './styles/app.css';

function App() {
  const { data, isLoading, isFetching, error, refetch } = useQuery({
    queryKey: ['users'],
    queryFn: fetchUsers,
    staleTime: 5000
  });

  return (
    <div className="container">
      <h1>React Query 캐싱 실습</h1>
      <p className="description">
        Node 서버에서 데이터를 조회하고, React Query의 캐시 및 재검증 구조를 확인한다.
      </p>

      <div className="panel">
        <button className="action-button" onClick={() => refetch()}>
          사용자 목록 다시 불러오기
        </button>

        {isLoading && <p className="status">최초 로딩 중입니다.</p>}
        {!isLoading && isFetching && (
          <p className="status">최신 데이터를 확인하는 중입니다.</p>
        )}
        {error && <p className="status error">{error.message}</p>}
      </div>

      {data && (
        <>
          <UserSummary />
          <UserList users={data} />
        </>
      )}
    </div>
  );
}

export default App;

Overwriting react-query-practice/client/src/App.jsx


In [ ]:
%%writefile react-query-practice/client/src/components/UserSummary.jsx

import { useQuery } from '@tanstack/react-query';
import { fetchUsers } from '../api/usersApi';

function UserSummary() {
  const { data } = useQuery({
    queryKey: ['users'],
    queryFn: fetchUsers,
    staleTime: 5000
  });

  return (
    <div className="panel">
      <h2>사용자 요약</h2>
      <p>총 인원: {data ? data.length : 0}명</p>
    </div>
  );
}

export default UserSummary;

Overwriting react-query-practice/client/src/components/UserSummary.jsx


In [ ]:
%%writefile react-query-practice/client/src/components/UserList.jsx

function UserList({ users }) {
  return (
    <div className="panel">
      <h2>사용자 목록</h2>

      <table className="user-table">
        <thead className="user-tableth">
          <tr>
            <th>번호</th>
            <th>이름</th>
            <th>부서</th>
            <th>갱신 시각</th>
          </tr>
        </thead>
        <tbody>
          {users.map(function (user) {
            return (
              <tr key={user.id}>
                <td className="user-tabletd">{user.id}</td>
                <td className="user-tabletd">{user.name}</td>
                <td className="user-tabletd">{user.department}</td>
                <td className="user-tabletd">{user.updatedAt}</td>
              </tr>
            );
          })}
        </tbody>
      </table>
    </div>
  );
}

export default UserList;

Overwriting react-query-practice/client/src/components/UserList.jsx


In [ ]:
%%writefile react-query-practice/client/src/styles/app.css

body {
  margin:0;
  font-family:Arial,sans-serif;
  background-color: #f5f6f8;
}

.container {
  max-width:960px;
  margin:0auto;
  padding:20px;
}

h1 {
  margin-bottom:10px;
}

.description {
  color: #666;
  margin-bottom:20px;
}

.panel {
  background: #ffffff;
  border-radius:8px;
  padding:16px;
  margin-bottom:16px;
  box-shadow:02px6pxrgba(0,0,0,0.1);
}

.action-button {
  padding:10px16px;
  border:none;
  background-color: #0064ff;
  color:white;
  border-radius:6px;
  cursor:pointer;
}

.action-button:hover {
  background-color: #004ed6;
}

.status {
  margin-top:10px;
  color: #333;
}

.status.error {
  color:red;
}

.user-table {
  width:100%;
  border-collapse:collapse;
  margin-top:10px;
}

.user-tableth,
.user-tabletd {
  border:1pxsolid #ddd;
  padding:8px;
}

.user-tableth {
  background-color: #f0f2f5;
}

Overwriting react-query-practice/client/src/styles/app.css


# 사용자 관리 실습

In [ ]:
%%writefile invalidation/server/server.js

const express = require('express');
const path = require('path');

const app = express();
const PORT = 5100;

app.use(express.json());
app.use(express.static(path.join(__dirname, 'static')));

let users = [
  { id: 1, name: '김민수' },
  { id: 2, name: '이서연' }
];

app.get('/api/users', function (req, res) {
  res.json(users);
});

app.post('/api/users', function (req, res) {
  const newUser = {
    id: Date.now(),
    name: req.body.name
  };

  users.push(newUser);

  res.json(newUser);
});

app.get('/*rest', function (req, res) {
  res.sendFile(path.join(__dirname, 'static', 'index.html'));
});

app.listen(PORT, function () {
  console.log('server running on port ' + PORT);
});

Overwriting invalidation/server/server.js


In [ ]:
%%writefile invalidation/vite/src/api/usersApi.js

export async function fetchUsers() {
  const response = await fetch('/api/users');

  if (!response.ok) {
    throw new Error('목록 조회 실패');
  }

  return await response.json();
}

export async function createUser(name) {
  const response = await fetch('/api/users', {
    method: 'POST',
    headers: {
      'Content-Type': 'application/json'
    },
    body: JSON.stringify({ name })
  });

  if (!response.ok) {
    throw new Error('사용자 생성 실패');
  }

  return await response.json();
}

Overwriting invalidation/vite/src/api/usersApi.js


In [ ]:
%%writefile invalidation/vite/src/main.jsx

import React from 'react';
import ReactDOM from 'react-dom/client';
import { QueryClient, QueryClientProvider } from '@tanstack/react-query';
import App from './App';

const queryClient = new QueryClient();

ReactDOM.createRoot(document.getElementById('root')).render(
  <QueryClientProvider client={queryClient}>
    <App />
  </QueryClientProvider>
);

Overwriting invalidation/vite/src/main.jsx


In [ ]:
%%writefile invalidation/vite/src/App.jsx

import { useQuery, useMutation, useQueryClient } from '@tanstack/react-query';
import { fetchUsers, createUser } from './api/usersApi';
import UserList from './components/UserList';
import UserForm from './components/UserForm';
import './styles/app.css';

function App() {
  const queryClient = useQueryClient();

  const { data, isLoading, isFetching, error } = useQuery({
    queryKey: ['users'],
    queryFn: fetchUsers
  });

  const mutation = useMutation({
    mutationFn: createUser,
    onSuccess: function () {
      queryClient.invalidateQueries({ queryKey: ['users'] });
    }
  });

  function handleAddUser(name) {
    mutation.mutate(name);
  }

  return (
    <div className="container">
      <h1 className="page-title">사용자 관리 실습</h1>
      <p className="page-description">
        Query Invalidation과 Loading, Fetching, Error 상태를 화면에서 구분하여 확인한다.
      </p>

      <div className="panel">
        <UserForm onAdd={handleAddUser} isPending={mutation.isPending} />

        {isLoading && (
          <div className="status-box status-loading">
            초기 데이터를 불러오는 중입니다.
          </div>
        )}

        {!isLoading && isFetching && (
          <div className="status-box status-fetching">
            목록을 최신 상태로 다시 확인하는 중입니다.
          </div>
        )}

        {mutation.isPending && (
          <div className="status-box status-mutation">
            사용자 정보를 서버에 저장하는 중입니다.
          </div>
        )}

        {error && (
          <div className="status-box status-error">
            오류가 발생했습니다: {error.message}
          </div>
        )}
      </div>

      <div className="panel">
        <h2 className="list-title">사용자 목록</h2>
        <UserList users={data || []} />
      </div>
    </div>
  );
}

export default App;

Overwriting invalidation/vite/src/App.jsx


In [ ]:
%%writefile invalidation/vite/src/components/UserForm.jsx

import { useState } from 'react';

function UserForm({ onAdd, isPending }) {
  const [name, setName] = useState('');

  function handleSubmit(event) {
    event.preventDefault();

    const trimmedName = name.trim();

    if (!trimmedName) {
      return;
    }

    onAdd(trimmedName);
    setName('');
  }

  return (
    <form onSubmit={handleSubmit} className="form-row">
      <input
        className="input-box"
        type="text"
        value={name}
        onChange={function (e) {
          setName(e.target.value);
        }}
        placeholder="추가할 사용자 이름 입력"
      />
      <button type="submit" className="action-button" disabled={isPending}>
        사용자 추가
      </button>
    </form>
  );
}

export default UserForm;

Overwriting invalidation/vite/src/components/UserForm.jsx


In [ ]:
%%writefile invalidation/vite/src/components/UserList.jsx

function UserList({ users }) {
  if (!users.length) {
    return <div className="empty-box">표시할 사용자 데이터가 없습니다.</div>;
  }

  return (
    <div className="user-list">
      {users.map(function (user) {
        return (
          <div key={user.id} className="user-item">
            <div className="user-name">{user.name}</div>
            <div className="user-id">ID: {user.id}</div>
          </div>
        );
      })}
    </div>
  );
}

export default UserList;

Overwriting invalidation/vite/src/components/UserList.jsx


In [ ]:
%%writefile invalidation/vite/src/styles/app.css

body {
  margin: 0;
  padding: 0;
  font-family: Arial, sans-serif;
  background-color: #f4f6f8;
  color: #222;
}

* {
  box-sizing: border-box;
}

#root {
  width: 100%;
}

.container {
  max-width: 960px;
  margin: 0 auto;
  padding: 40px 20px;
}

.page-title {
  margin: 0 0 12px 0;
  font-size: 28px;
  font-weight: bold;
}

.page-description {
  margin: 0 0 24px 0;
  line-height: 1.6;
  color: #555;
}

.panel {
  background-color: #ffffff;
  border: 1px solid #d9dee5;
  border-radius: 12px;
  padding: 20px;
  margin-bottom: 20px;
}

.form-row {
  display: flex;
  gap: 12px;
  align-items: center;
}

.input-box {
  flex: 1;
  height: 44px;
  padding: 0 14px;
  border: 1px solid #cfd6df;
  border-radius: 8px;
  font-size: 15px;
  outline: none;
}

.input-box:focus {
  border-color: #2563eb;
}

.action-button {
  height: 44px;
  padding: 0 18px;
  border: none;
  border-radius: 8px;
  background-color: #2563eb;
  color: #ffffff;
  font-size: 15px;
  cursor: pointer;
}

.action-button:disabled {
  background-color: #9fb8f5;
  cursor: not-allowed;
}

.status-box {
  margin-top: 16px;
  padding: 12px 14px;
  border-radius: 8px;
  font-size: 14px;
  line-height: 1.5;
}

.status-loading {
  background-color: #eef4ff;
  color: #1d4ed8;
  border: 1px solid #cfe0ff;
}

.status-fetching {
  background-color: #f5faff;
  color: #0f766e;
  border: 1px solid #c7ece8;
}

.status-error {
  background-color: #fff1f1;
  color: #c62828;
  border: 1px solid #f3c1c1;
}

.status-mutation {
  background-color: #fff8e8;
  color: #9a6700;
  border: 1px solid #f1ddb0;
}

.list-title {
  margin: 0 0 16px 0;
  font-size: 20px;
  font-weight: bold;
}

.user-list {
  display: flex;
  flex-direction: column;
  gap: 12px;
}

.user-item {
  padding: 14px 16px;
  border: 1px solid #e3e8ee;
  border-radius: 10px;
  background-color: #fafbfc;
}

.user-name {
  font-size: 16px;
  font-weight: bold;
  margin-bottom: 6px;
}

.user-id {
  font-size: 13px;
  color: #666;
}

.empty-box {
  padding: 16px;
  border: 1px dashed #c8d0da;
  border-radius: 10px;
  color: #666;
  background-color: #fafbfc;
}

Overwriting invalidation/vite/src/styles/app.css


# React Query 사용자 추가 실습

In [28]:
%%writefile react-query-practice/server/server.js

const express = require('express');
const path = require('path');

const app = express();
const PORT = 5600;

app.use(express.json());
app.use(express.static(path.join(__dirname, 'static')));

let users = [
  { id: 1, name: '김민수' },
  { id: 2, name: '이서연' }
];

app.get('/api/users', function (req, res) {
  res.json(users);
});

app.post('/api/users', function (req, res) {
  const newUser = {
    id: Date.now(),
    name: req.body.name
  };

  users.push(newUser);

  res.json(newUser);
});

app.get('/*rest', function (req, res) {
  res.sendFile(path.join(__dirname, 'static', 'index.html'));
});

app.listen(PORT, function () {
  console.log('Node server is running on port ' + PORT);
});

Overwriting react-query-practice/server/server.js


In [29]:
%%writefile react-query-practice/vite/src/api/usersApi.js

export async function fetchUsers() {
  const response = await fetch('/api/users');

  if (!response.ok) {
    throw new Error('목록 조회 실패');
  }

  return await response.json();
}

export async function createUser(name) {
  const response = await fetch('/api/users', {
    method: 'POST',
    headers: {
      'Content-Type': 'application/json'
    },
    body: JSON.stringify({ name })
  });

  if (!response.ok) {
    throw new Error('사용자 생성 실패');
  }

  return await response.json();
}

Overwriting react-query-practice/vite/src/api/usersApi.js


In [37]:
%%writefile react-query-practice/vite/src/App.jsx

import { useQuery, useMutation, useQueryClient } from '@tanstack/react-query';
import { fetchUsers, createUser } from './api/usersApi';
import UserList from './components/UserList';
import UserForm from './components/UserForm';

function App() {
  const queryClient = useQueryClient();

  const { data, isLoading, error } = useQuery({
    queryKey: ['users'],
    queryFn: fetchUsers
  });

  const mutation = useMutation({
    mutationFn: createUser,

    onMutate: async function (newName) {
      await queryClient.cancelQueries({ queryKey: ['users'] });

      const previous = queryClient.getQueryData(['users']);

      queryClient.setQueryData(['users'], function (old) {
        return [
          ...old,
          { id: Date.now(), name: newName }
        ];
      });

      return { previous };
    },

    onError: function (error, newName, context) {
      queryClient.setQueryData(['users'], context.previous);
    },

    onSettled: function () {
      queryClient.invalidateQueries({ queryKey: ['users'] });
    }
  });

  function handleAdd(name) {
    mutation.mutate(name);
  }

  if (isLoading) return <p>로딩 중</p>;
  if (error) return <p>오류 발생</p>;

  return (
    <div>
      <UserForm onAdd={handleAdd} />

      {mutation.isPending && <p>저장 중</p>}

      <UserList users={data} />
    </div>
  );
}

export default App;

Overwriting react-query-practice/vite/src/App.jsx


In [31]:
%%writefile react-query-practice/vite/src/main.jsx

import React from 'react';
import ReactDOM from 'react-dom/client';
import { QueryClient, QueryClientProvider } from '@tanstack/react-query';
import App from './App';

const queryClient = new QueryClient();

ReactDOM.createRoot(document.getElementById('root')).render(
  <QueryClientProvider client={queryClient}>
    <App />
  </QueryClientProvider>
);

Overwriting react-query-practice/vite/src/main.jsx


In [32]:
%%writefile react-query-practice/vite/src/components/UserList.jsx

function UserList({ users }) {
  if (!users.length) {
    return <div className="empty-box">표시할 사용자 데이터가 없습니다.</div>;
  }

  return (
    <div className="user-list">
      {users.map(function (user) {
        return (
          <div key={user.id} className="user-item">
            <div className="user-name">{user.name}</div>
            <div className="user-id">ID: {user.id}</div>
          </div>
        );
      })}
    </div>
  );
}

export default UserList;

Overwriting react-query-practice/vite/src/components/UserList.jsx


In [33]:
%%writefile react-query-practice/vite/src/components/UserForm.jsx

import { useState } from 'react';

function UserForm({ onAdd, isPending }) {
  const [name, setName] = useState('');

  function handleSubmit(event) {
    event.preventDefault();

    const trimmedName = name.trim();

    if (!trimmedName) {
      return;
    }

    onAdd(trimmedName);
    setName('');
  }

  return (
    <form onSubmit={handleSubmit} className="form-row">
      <input
        className="input-box"
        type="text"
        value={name}
        onChange={function (e) {
          setName(e.target.value);
        }}
        placeholder="추가할 사용자 이름 입력"
      />
      <button type="submit" className="action-button" disabled={isPending}>
        사용자 추가
      </button>
    </form>
  );
}

export default UserForm;

Overwriting react-query-practice/vite/src/components/UserForm.jsx
